In [7]:
from openai import OpenAI

In [8]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

In [9]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': 'hai',
        }
    ],
    model='llama3.1',
)
print(chat_completion.choices[0].message.content)

Hai! Bagaimana hari kamu?


In [10]:
from my_agent.utils import OpenAIGPT

d:\AI\simpleai\my_agent\utils\llm_service.py:179: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  human_content = f"""The output you given has issue while parsing to json, re.search(r'^\s*\{{[\s\S]*\}}\s*$', your_output) couldn't match any match:
d:\AI\simpleai\my_agent\utils\llm_service.py:179: SyntaxWarning: "\}" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\}"? A raw string is also an option.
  human_content = f"""The output you given has issue while parsing to json, re.search(r'^\s*\{{[\s\S]*\}}\s*$', your_output) couldn't match any match:
d:\AI\simpleai\my_agent\utils\llm_service.py:182: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  please try one more time."""
d:\AI\simpleai\my_agent\utils\llm_service.py:182: SyntaxWarning: "\s" is an 

In [11]:
from dotenv import load_dotenv
load_dotenv()

llm = OpenAIGPT()

In [12]:
from langchain_core.messages import HumanMessage
for token in llm.stream("hai"):
    print(token.content, end='', flush=True)

Hai! Bagaimana hari Anda?

In [13]:
from typing import TypedDict
from pydantic import BaseModel
class state(BaseModel):
    answer: str
    justification: str

In [14]:
state.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'justification': {'title': 'Justification', 'type': 'string'}},
 'required': ['answer', 'justification'],
 'title': 'state',
 'type': 'object'}

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool
convert_to_openai_tool(state)

{'type': 'function',
 'function': {'name': 'state',
  'description': '',
  'parameters': {'properties': {'answer': {'type': 'string'},
    'justification': {'type': 'string'}},
   'required': ['answer', 'justification'],
   'type': 'object'}}}

In [16]:
struct_llm = llm.with_structured_output(state)

In [17]:
class State(BaseModel):
    vajid: str
    good: str

In [18]:
struct_llm.invoke('which coutry have done most wars')

AIMessage(content='{"answer":"United States of America","justification":"According to various sources, the United States has involved itself in numerous conflicts and wars throughout its history, including World War I, World War II, Korea, Vietnam, Gulf Wars, Afghanistan, among others. This makes it one of the most intervened countries globally."}', additional_kwargs={'parsed': state(answer='United States of America', justification='According to various sources, the United States has involved itself in numerous conflicts and wars throughout its history, including World War I, World War II, Korea, Vietnam, Gulf Wars, Afghanistan, among others. This makes it one of the most intervened countries globally.')}, response_metadata={}, id='lc_run--019ea7c2-1005-7063-99e6-b641659d43fd-0', tool_calls=[], invalid_tool_calls=[])

In [19]:
from langchain_core.output_parsers import JsonOutputParser
text = """so tha is the import the {"answer": "and output"}"""
parser = JsonOutputParser()
parser.invoke('{"hai": "hallow"}')

{'hai': 'hallow'}

In [20]:
from langchain_core.output_parsers import PydanticOutputParser

In [21]:
parser = PydanticOutputParser(pydantic_object=state)

In [22]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"title": "Answer", "type": "string"}, "justification": {"title": "Justification", "type": "string"}}, "required": ["answer", "justification"]}
```


In [71]:
from langchain_core.tools import tool

@tool
def calculator(expression: str):
    """Calculate any Expression like '2 + 5' or 2 / 6 etc."""
    return eval(expression)

In [72]:
tool_llm = llm.bind_tools([calculator])

In [78]:
messages = [
    HumanMessage(content='what is 5th multiply of 10 and multiply it with 400 and subtract 3 from it')
]
out = tool_llm.invoke(messages)
out.tool_calls

[{'name': 'calculator',
  'args': {'expression': '5*10*400-3'},
  'id': '3c49f1c4-25d3-48c6-808b-bf523c9dca9f',
  'type': 'tool_call'}]

In [79]:
tool_out = calculator.invoke(out.tool_calls[0]['args'])

In [80]:
from langchain_core.messages import ToolCall, ToolMessage, AIMessage
messages.append(ToolMessage(content=f'{tool_out}',tool_call_id=out.tool_calls[0]['id']))
print(llm.invoke(messages).content)

To solve this problem, we need to follow the order of operations (PEMDAS):

1. Calculate the 5th multiple of 10:
   10 × 1 = 10
   10 × 2 = 20
   10 × 3 = 30
   10 × 4 = 40
   10 × 5 = 50

   So, the 5th multiple of 10 is 50.

2. Multiply it with 400:
   50 × 400 = 20000

3. Subtract 3 from it:
   20000 - 3 = 19997

Therefore, the result of the given problem is indeed 19997.


In [81]:
from langchain_core.messages import convert_to_openai_messages
convert_to_openai_messages(messages)

[{'role': 'user',
  'content': 'what is 5th multiply of 10 and multiply it with 400 and subtract 3 from it'},
 {'role': 'tool',
  'tool_call_id': '3c49f1c4-25d3-48c6-808b-bf523c9dca9f',
  'content': '19997'}]